In [5]:
import chromadb
import pandas as pd
import os

# 1. Configuration
db_path = os.path.abspath(r"D:\pY\InsuranceRAG\local_db")
print(f"📂 Looking for database at: {db_path}")

client = chromadb.PersistentClient(path=db_path)
collections = client.list_collections()
collection_names = [c.name for c in collections]
print(f"📚 Found {len(collections)} collections: {collection_names}")

# 2. Function to inspect a specific collection
def inspect_collection_dynamic(name):
    if name not in collection_names:
        print(f"❌ Collection '{name}' does not exist.")
        return None
    
    collection = client.get_collection(name=name)
    results = collection.get(include=["documents", "metadatas"])
    
    if not results['ids']:
        print(f"⚠️ Collection '{name}' exists but is empty.")
        return None
        
    records = []
    for i in range(len(results['ids'])):
        # 1. Start with the core fields (ID and Document Preview)
        entry = {
            "ID": results['ids'][i],
            "Content Preview": results['documents'][i][:100].replace('\n', ' ') + "..."
        }
        
        # 2. Dynamically add all metadata fields
        meta = results['metadatas'][i] if results['metadatas'] else {}
        for key, value in meta.items():
            # If the value is a file path, just show the filename for neatness
            if key == "source" and isinstance(value, str):
                entry["Source File"] = os.path.basename(value)
            else:
                entry[key] = value
                
        records.append(entry)
    
    # 3. Create DataFrame (Pandas handles the alignment of different keys automatically)
    df = pd.DataFrame(records)
    
    # Optional: Move 'ID' and 'Content Preview' to the end for better readability
    cols = [c for c in df.columns if c not in ["ID", "Content Preview"]] + ["ID", "Content Preview"]
    return df[cols]

# 3. Fetch and Display Both Collections
print("\n--- 📜 MASTER POLICIES ---")
df_policies = inspect_collection_dynamic("policy_master_collection")
if df_policies is not None:
    display(df_policies.sort_values(by="Source File"))

print("\n--- 📑 CLIENT CLAIMS ---")
df_claims = inspect_collection_dynamic("claims_collection")
if df_claims is not None:
    display(df_claims)

print("\n--- 📑 EVALUATION AUDIT LOG ---")
df_audit = inspect_collection_dynamic("evaluation_audit_log")
if df_audit is not None:
    display(df_audit)

print("\n--- 📑 INSTANCE LOG ---")
df_instance_log = inspect_collection_dynamic("instance_log")
if df_instance_log is not None:
    display(df_instance_log)

# 4. Summary Table
if df_policies is not None or df_claims is not None:
    print("\n📊 Database Summary:")
    for c in collections:
        print(f" - {c.name}: {c.count()} chunks")

📂 Looking for database at: D:\pY\InsuranceRAG\local_db
📚 Found 4 collections: ['claims_collection', 'evaluation_audit_log', 'instance_log', 'policy_master_collection']

--- 📜 MASTER POLICIES ---


,trapped,modDate,document_category,creator,format,subject,source_type,keywords,creationDate,author,producer,total_pages,page,moddate,Source File,title,creationdate,file_path,ID,Content Preview
154,,D:20220707173344+04'00',Helath,Adobe InDesign 15.1 (Macintosh),PDF 1.4,,Policy,,D:20220707173342+04'00',,Adobe PDF Library 15.0,18,17,2022-07-07T17:33:44+04:00,Membership Handbook.pdf,,2022-07-07T17:33:42+04:00,D:\pY\InsuranceRAG\storage\policies\Helath\202...,620cb565-fc58-4acd-9a74-90d28904e4ae,(6) of 2007 concerning the establishment of In...
128,,D:20220707173344+04'00',Helath,Adobe InDesign 15.1 (Macintosh),PDF 1.4,,Policy,,D:20220707173342+04'00',,Adobe PDF Library 15.0,18,7,2022-07-07T17:33:44+04:00,Membership Handbook.pdf,,2022-07-07T17:33:42+04:00,D:\pY\InsuranceRAG\storage\policies\Helath\202...,3546c24a-9c78-4631-af31-77df794614c2,When we use these terms they are in bold prin...
127,,D:20220707173344+04'00',Helath,Adobe InDesign 15.1 (Macintosh),PDF 1.4,,Policy,,D:20220707173342+04'00',,Adobe PDF Library 15.0,18,7,2022-07-07T17:33:44+04:00,Membership Handbook.pdf,,2022-07-07T17:33:42+04:00,D:\pY\InsuranceRAG\storage\policies\Helath\202...,251f04ce-e865-4cc6-8309-2e73da7e4862,Policy Handbook Health Insurance 14 15 to be ...
126,,D:20220707173344+04'00',Helath,Adobe InDesign 15.1 (Macintosh),PDF 1.4,,Policy,,D:20220707173342+04'00',,Adobe PDF Library 15.0,18,6,2022-07-07T17:33:44+04:00,Membership Handbook.pdf,,2022-07-07T17:33:42+04:00,D:\pY\InsuranceRAG\storage\policies\Helath\202...,f0dc7cdd-6f6c-46cd-b850-94d985d97ec8,Please also note that no claim of any kind wi...
125,,D:20220707173344+04'00',Helath,Adobe InDesign 15.1 (Macintosh),PDF 1.4,,Policy,,D:20220707173342+04'00',,Adobe PDF Library 15.0,18,6,2022-07-07T17:33:44+04:00,Membership Handbook.pdf,,2022-07-07T17:33:42+04:00,D:\pY\InsuranceRAG\storage\policies\Helath\202...,9987c52a-8572-4306-a043-81e16a84abd6,What happens if you wish to cancel your poli...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27,,D:20220616133420+04'00',Travel,Adobe InDesign 15.1 (Macintosh),PDF 1.4,,Policy,,D:20220616133418+04'00',,Adobe PDF Library 15.0,18,10,2022-06-16T13:34:20+04:00,TRAVEL_TC_HSBCUAE_EN.pdf,,2022-06-16T13:34:18+04:00,D:\pY\InsuranceRAG\storage\policies\Travel\202...,cf669d4a-e4b2-432e-b8a6-84b37fad71c1,What is not covered a)\t Your failure to check...
26,,D:20220616133420+04'00',Travel,Adobe InDesign 15.1 (Macintosh),PDF 1.4,,Policy,,D:20220616133418+04'00',,Adobe PDF Library 15.0,18,10,2022-06-16T13:34:20+04:00,TRAVEL_TC_HSBCUAE_EN.pdf,,2022-06-16T13:34:18+04:00,D:\pY\InsuranceRAG\storage\policies\Travel\202...,6112f350-86ab-4b5b-a951-4bbecba64146,Hijack of the Insured Person or of any person...
25,,D:20220616133420+04'00',Travel,Adobe InDesign 15.1 (Macintosh),PDF 1.4,,Policy,,D:20220616133418+04'00',,Adobe PDF Library 15.0,18,10,2022-06-16T13:34:20+04:00,TRAVEL_TC_HSBCUAE_EN.pdf,,2022-06-16T13:34:18+04:00,D:\pY\InsuranceRAG\storage\policies\Travel\202...,7781282d-7a74-4ba7-a287-c4c9c7dc45db,Policy Handbook Travel Smart 20 21 bail bond ...
34,,D:20220616133420+04'00',Travel,Adobe InDesign 15.1 (Macintosh),PDF 1.4,,Policy,,D:20220616133418+04'00',,Adobe PDF Library 15.0,18,13,2022-06-16T13:34:20+04:00,TRAVEL_TC_HSBCUAE_EN.pdf,,2022-06-16T13:34:18+04:00,D:\pY\InsuranceRAG\storage\policies\Travel\202...,7115c50e-730b-4c87-a954-eaa12f69126d,Policy Handbook Travel Smart 26 27 8 Gener...



--- 📑 CLIENT CLAIMS ---
⚠️ Collection 'claims_collection' exists but is empty.

--- 📑 EVALUATION AUDIT LOG ---


,status,instance_id,ID,Content Preview
0,ERROR,f1cfd046-8dcb-4531-ab3f-85a107324240,df7f86b4-4a04-4c35-a105-de4f865fade8,Critical Graph Failure: unsupported operand ty...
1,ERROR,cce39e15-d794-4767-a99b-b1594cbf6378,693fac93-887a-43e8-9b18-d506bb251038,Critical Graph Failure: unsupported operand ty...



--- 📑 INSTANCE LOG ---


,client_id,status,instance_id,error_details,completion_time,ID,Content Preview
0,12345,FAILED,f1cfd046-8dcb-4531-ab3f-85a107324240,unsupported operand type(s) for +: 'dict' and ...,2026-02-11T02:02:08.118007,f1cfd046-8dcb-4531-ab3f-85a107324240,Investigation terminated: unsupported operand ...
1,12345,FAILED,cce39e15-d794-4767-a99b-b1594cbf6378,unsupported operand type(s) for +: 'dict' and ...,2026-02-11T02:09:01.635914,cce39e15-d794-4767-a99b-b1594cbf6378,Investigation terminated: unsupported operand ...



📊 Database Summary:
 - claims_collection: 0 chunks
 - evaluation_audit_log: 2 chunks
 - instance_log: 2 chunks
 - policy_master_collection: 155 chunks
